# Combo 4 - Chakra ViT training from scratch on the NEW dataset (1000 images + 1000 masks)

**Before running:** create a Kaggle Dataset containing `images/` and `masks/` folders (same base filename for each pair, e.g. `img001.jpg` + `img001.png` or `img001_mask.png`), attach it via *Add Input*, and **detach the old `chakra-leakbench-v1` dataset** (it caused the 6523 vs 2248 mismatch).

**Expected run time on a T4** (first run: 19 epochs in ~12 h = 37.9 min/epoch on the old data; the exact old per-epoch sample count isn't recorded, so two cases):

| Old epochs actually used | Per-image cost | 1000 images / epoch | 30 epochs |
|---|---|---|---|
| all 6523 images | 0.35 s | ~6 min | **~3 h** |
| only the 2248 masked pairs | 1.01 s | ~17 min | **~8.5 h** |

Either way it fits inside the 12 h limit. The run cell prints a live minutes/epoch and projected total after every epoch, so you will know the real number after epoch 1.

In [ ]:
!git clone -q https://github.com/varshasrao58/chakramodel.git /kaggle/working/chakramodel
%cd /kaggle/working/chakramodel
!pip install -q timm==0.9.16 opencv-python-headless
import os, glob, shutil, subprocess
assert os.path.exists('/kaggle/input'), 'Run this notebook on Kaggle with GPU enabled'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode())

## Step 1 - Load and validate the new dataset
Pairs images with masks **by filename**, reports anything unmatched, checks image/mask sizes, then copies only the matched pairs to the folder the training script reads. You want to see `matched pairs: 1000` and no warnings.

In [ ]:
import os, re, shutil
from pathlib import Path
from PIL import Image

# ---------------- EDIT THIS ----------------
DATASET_ROOT = None   # e.g. '/kaggle/input/datasets/<your-username>/<new-dataset>' ; None = auto-detect under /kaggle/input
# -------------------------------------------
INPUT_ROOT = Path(os.environ.get('INPUT_ROOT', '/kaggle/input'))
TARGET = Path(os.environ.get('TARGET_ROOT', '/kaggle/working/chakramodel/src/data/kvasir-seg'))   # folder the training script reads
EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def dirs_named(root, name):
    return [p for p in Path(root).rglob('*') if p.is_dir() and p.name.lower() == name]

root = Path(DATASET_ROOT) if DATASET_ROOT else None
if root is None:
    cands = sorted({p.parent for p in dirs_named(INPUT_ROOT, 'images')} & {p.parent for p in dirs_named(INPUT_ROOT, 'masks')})
    if len(cands) != 1:
        raise RuntimeError(f'Expected exactly one folder containing images/ and masks/ under {INPUT_ROOT}, found {len(cands)}: {cands}. '
                           'Detach old datasets or set DATASET_ROOT.')
    root = cands[0]
img_dirs, mask_dirs = dirs_named(root, 'images'), dirs_named(root, 'masks')
if root.name.lower() in ('images', 'masks'): raise RuntimeError('Point DATASET_ROOT at the folder that CONTAINS images/ and masks/')
print('dataset root:', root)

def files(dirs): return [f for d in dirs for f in d.rglob('*') if f.is_file() and f.suffix.lower() in EXT]
def norm(stem):  # tolerate masks named  xxx_mask / xxx-mask / xxx_segmentation / xxx_gt
    return re.sub(r'[_\-](mask|segmentation|seg|gt|label)$', '', stem.lower())

imgs, masks = files(img_dirs), files(mask_dirs)
by_img, by_mask = {}, {}
for f in imgs:  by_img.setdefault(norm(f.stem), []).append(f)
for f in masks: by_mask.setdefault(norm(f.stem), []).append(f)
dup = [k for d in (by_img, by_mask) for k, v in d.items() if len(v) > 1]
if dup: raise RuntimeError(f'{len(dup)} duplicate names across subfolders (e.g. {dup[:5]}) - flattening would overwrite files')

pairs = sorted(set(by_img) & set(by_mask))
only_img, only_mask = sorted(set(by_img) - set(by_mask)), sorted(set(by_mask) - set(by_img))
print(f'images found: {len(imgs)} | masks found: {len(masks)} | matched pairs: {len(pairs)}')
print(f'images without a mask: {len(only_img)} {only_img[:5]} | masks without an image: {len(only_mask)} {only_mask[:5]}')
if only_img or only_mask:
    print('WARNING: unmatched files are skipped - fix the dataset if you expected 1000/1000')

bad_size, empty = [], 0
for k in pairs:
    with Image.open(by_img[k][0]) as a, Image.open(by_mask[k][0]) as b:
        if a.size != b.size: bad_size.append((k, a.size, b.size))
        if b.convert('L').getextrema()[1] == 0: empty += 1
print(f'size mismatches (image vs mask): {len(bad_size)} {bad_size[:3]} | all-black masks: {empty}')
if bad_size: raise RuntimeError('Some image/mask pairs have different sizes - fix those files first')

ti, tm = TARGET / 'images', TARGET / 'masks'
for d in (ti, tm):
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True, exist_ok=True)
exts = set()
for k in pairs:
    i, m = by_img[k][0], by_mask[k][0]
    shutil.copy(i, ti / i.name)
    shutil.copy(m, tm / (i.stem + m.suffix))     # mask gets exactly the image's base name
    exts.add((i.suffix.lower(), m.suffix.lower()))
print(f'copied {len(pairs)} pairs -> {TARGET}   (image/mask extension combos: {sorted(exts)})')
assert len(list(ti.glob('*')) ) == len(list(tm.glob('*'))) == len(pairs) > 0

## Step 2 (optional but recommended) - Per-epoch resumable checkpoint
Patches `train_transformer.py` to save `/kaggle/working/ckpt/last.pth` after every epoch, so a timeout never costs you the run again. If it can't recognise the script, it changes nothing and training still works.

In [ ]:
SCRIPT_PATH = '/kaggle/working/chakramodel/src/chakra_transformer/train_transformer.py'
import re, shutil, py_compile, difflib
from pathlib import Path

SCRIPT = Path(SCRIPT_PATH)
BACKUP = Path(str(SCRIPT) + '.orig')
MARK = '# >>> RESUME PATCH'

def stmt_end(lines, i):
    """index of the last line of the (possibly multi-line) statement starting at lines[i]"""
    depth = 0
    for j in range(i, len(lines)):
        depth += sum(lines[j].count(c) for c in '([{') - sum(lines[j].count(c) for c in ')]}')
        if depth <= 0:
            return j
    return i

def block(ind, text):
    return [ind + l if l.strip() else l for l in text.strip('\n').split('\n')]

src = SCRIPT.read_text()
if MARK not in src and not BACKUP.exists():
    shutil.copy(SCRIPT, BACKUP)
if MARK in src:
    print('Script is already patched - nothing to do.')
else:
    lines = src.split('\n')
    problems, notes = [], []

    # 1) optimizer line -> gives us the model + optimizer variable names
    oi = next((i for i, l in enumerate(lines)
               if re.match(r'^\s*(\w+)\s*=\s*[\w\.]*(Adam|AdamW|SGD|RAdam)\w*\(', l)), None)
    if oi is None:
        problems.append('could not find the line that creates the optimizer (expected  opt = ...Adam(model.parameters(), ...))')
    else:
        m = re.match(r'^(\s*)(\w+)\s*=', lines[oi])
        ind, OPT = m.group(1), m.group(2)
        pm = re.search(r'([\w\.]+)\.parameters\(\)', '\n'.join(lines[oi:stmt_end(lines, oi) + 1]))
        if not pm:
            problems.append('optimizer is not built from <model>.parameters() (maybe param groups) - cannot tell which variable is the model')
        else:
            MODEL = pm.group(1)

    # 2) epoch loop
    li = next((i for i, l in enumerate(lines) if re.match(r'^\s*for\s+(\w+)\s+in\s+range\(', l) and 'epoch' in l.lower()), None)
    if li is None:
        problems.append('could not find the epoch loop (expected  for epoch in range(N):)')
    else:
        lm = re.match(r'^(\s*for\s+(\w+)\s+in\s+)range\((.*)\)\s*:\s*$', lines[li])
        if not lm or ',' in lm.group(3):
            problems.append(f'epoch loop is not a simple range(N): {lines[li].strip()!r}')
        else:
            EPV = lm.group(2)

    # 3) epoch summary print line (to attach the per-epoch checkpoint save)
    pi = next((i for i, l in enumerate(lines) if 'Epoch [' in l and 'print' in l), None)
    if pi is None:
        problems.append('could not find the per-epoch print ("Epoch [ ...")')

    if problems:
        print('AUTO-PATCH NOT APPLIED - nothing was changed. Reasons:')
        for p in problems: print('  -', p)
        print('\nRelevant lines of the script (paste these to Claude if you want an exact patch):')
        for i, l in enumerate(lines):
            if re.search(r'optim|scheduler|for .* in range|Epoch \[|best|torch\.save|load_state_dict|DataParallel', l):
                print(f'{i+1:4d}: {l}')
        print('\nContinuing WITHOUT per-epoch checkpoints (training itself is unaffected).')
    else:
        edits = []   # (line_index, 'before'|'after'|'replace', new_lines)
        # model load, before optimizer creation
        edits.append((oi, 'before', block(ind, f'''
{MARK} (load weights)
import os as _os, torch as _torch
_rk = {{}}
if _os.environ.get('RESUME_CKPT'):
    _ck = _torch.load(_os.environ['RESUME_CKPT'], map_location='cpu')
    _sd = _ck
    if isinstance(_ck, dict):
        for _k in ('model', 'model_state_dict', 'state_dict'):
            if isinstance(_ck.get(_k), dict):
                _sd = _ck[_k]; break
    _sd = {{(k[7:] if k.startswith('module.') else k): v for k, v in _sd.items()}}
    _tgt = {MODEL}.module if hasattr({MODEL}, 'module') else {MODEL}
    _res = _tgt.load_state_dict(_sd, strict=False)
    print(f"[resume] weights loaded from {{_os.environ['RESUME_CKPT']}} | missing={{len(_res.missing_keys)}} unexpected={{len(_res.unexpected_keys)}}")
    if len(_res.missing_keys) > 0.05 * len(_tgt.state_dict()):
        raise RuntimeError('Checkpoint does not match the model (too many missing keys) - refusing to train from a half-loaded model')
    _rk = _ck if isinstance(_ck, dict) else {{}}
''')))
        oe = stmt_end(lines, oi)
        # optimizer state (only present in last.pth), after optimizer creation
        after_opt = block(ind, f'''
if isinstance(_rk.get('optimizer'), dict):
    _lr0 = [g['lr'] for g in {OPT}.param_groups]
    {OPT}.load_state_dict(_rk['optimizer'])
    for _g, _l in zip({OPT}.param_groups, _lr0): _g['lr'] = _l  # keep base LR so the schedule fast-forward below is exact
    print('[resume] optimizer state restored')
''')
        # scheduler fast-forward
        si = next((i for i, l in enumerate(lines) if re.match(
            r'^\s*(\w+)\s*=\s*[\w\.]*(CosineAnnealingLR|StepLR|MultiStepLR|ExponentialLR|LambdaLR)\(', l)), None)
        if si is not None:
            sm = re.match(r'^(\s*)(\w+)\s*=', lines[si]); SCH = sm.group(2); sind = sm.group(1)
            se = stmt_end(lines, si)
            sched_block = block(sind, f'''
for _ in range(int(_os.environ.get('RESUME_START', '0'))): {SCH}.step()
print('[resume] LR after fast-forwarding schedule:', {OPT}.param_groups[0]['lr'])
''')
            if se == oe: after_opt += sched_block
            else: edits.append((se, 'after', sched_block))
        else:
            notes.append('no epoch-stepped LR scheduler found - LR is probably computed from the epoch inside the loop, which the loop change below handles')
        edits.append((oe, 'after', after_opt))
        # loop start
        lm = re.match(r'^(\s*for\s+(\w+)\s+in\s+)range\((.*)\)\s*:\s*$', lines[li])
        edits.append((li, 'replace', [f"{lm.group(1)}range(int(__import__('os').environ.get('RESUME_START', '0')), {lm.group(3)}):"]))
        # best-metric initial value
        bi = next((i for i, l in enumerate(lines) if re.match(
            r'^\s*best\w*\s*=\s*(0|0\.0|-1|-1\.0|float\([\'"]-inf[\'"]\))\s*$', l)), None)
        if bi is not None:
            bm = re.match(r'^(\s*)(best\w*)\s*=\s*(.*)$', lines[bi])
            edits.append((bi, 'replace', [f"{bm.group(1)}{bm.group(2)} = float(__import__('os').environ.get('RESUME_BEST', {bm.group(3)}))"]))
        else:
            notes.append('best-metric variable not found - a worse epoch could overwrite chakra_transformer_best.pth (a backup is made by the notebook before training)')
        # per-epoch resumable checkpoint, after the epoch print
        pe = stmt_end(lines, pi); pind = re.match(r'^(\s*)', lines[pi]).group(1)
        edits.append((pe, 'after', block(pind, f'''
{MARK} (resumable checkpoint every epoch, outside the repo)
_t = {MODEL}.module if hasattr({MODEL}, 'module') else {MODEL}
__import__('os').makedirs('/kaggle/working/ckpt', exist_ok=True)
__import__('torch').save({{'model': _t.state_dict(), 'optimizer': {OPT}.state_dict(), 'completed_epochs': {EPV} + 1}}, '/kaggle/working/ckpt/last.tmp')
__import__('os').replace('/kaggle/working/ckpt/last.tmp', '/kaggle/working/ckpt/last.pth')
''')))
        # apply from the bottom up so indices stay valid
        _order = {'after': 0, 'replace': 1, 'before': 2}   # same-line ties: after first, before last
        for idx, kind, new in sorted(edits, key=lambda e: (-e[0], _order[e[1]])):
            if kind == 'before':  lines[idx:idx] = new
            elif kind == 'after': lines[idx+1:idx+1] = new
            else:                 lines[idx:idx+1] = new
        SCRIPT.write_text('\n'.join(lines))
        try:
            py_compile.compile(str(SCRIPT), doraise=True)
        except py_compile.PyCompileError as e:
            shutil.copy(BACKUP, SCRIPT)
            raise RuntimeError(f'Patched script has a syntax error, original restored: {e}')
        print('Patched OK. Diff vs original:\n')
        print('\n'.join(difflib.unified_diff(src.split('\n'), lines, 'original', 'patched', lineterm='', n=1)))
        for n in notes: print('NOTE:', n)

## Step 3 - Train
Same settings as the first run (`--epochs 30 --batch-size 8`, ImageNet-initialised ViT). Watch the `[timer]` lines.

In [ ]:
%cd /kaggle/working/chakramodel
import subprocess, sys, os, re, time

EPOCHS, BATCH_SIZE = 30, 8
TEACHER_CKPT = ''      # '' = ImageNet-initialised ViT, same as the first run. Only set a path if you deliberately want a different init.

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/chakramodel:/kaggle/working/chakramodel/src:' + env.get('PYTHONPATH', '')
env['PYTHONUNBUFFERED'] = '1'
env['RESUME_START'] = '0'          # fresh run (only used if the optional patch above was applied)

command = [sys.executable, '/kaggle/working/chakramodel/src/chakra_transformer/train_transformer.py',
           '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE)]
if TEACHER_CKPT: command += ['--teacher-checkpoint', TEACHER_CKPT]

SESSION_LIMIT_H = 12
t0 = time.time()
proc = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'Epoch \[\s*(\d+)\s*/\s*(\d+)\s*\]', line)
    if m:
        n, tot = int(m.group(1)), int(m.group(2)); el = time.time() - t0
        proj = el / n * tot / 3600
        flag = 'ok' if proj < SESSION_LIMIT_H - 0.5 else 'WARNING: projected to hit the 12 h session limit - lower EPOCHS or resume from ckpt/last.pth'
        print(f'    [timer] {el/60:5.1f} min elapsed | {el/n/60:5.1f} min/epoch | ~{(tot-n)*el/n/3600:4.2f} h left | projected total {proj:4.2f} h | {flag}')
if proc.wait() != 0:
    raise subprocess.CalledProcessError(proc.returncode, command)

## Step 4 - Collect outputs
Everything under `/kaggle/working` is kept when you use **Save Version -> Save & Run All**.

In [ ]:
import shutil, os
from pathlib import Path
Path('/kaggle/working/ckpt').mkdir(exist_ok=True, parents=True)
best = '/kaggle/working/chakramodel/src/weights/chakra_transformer_best.pth'
if os.path.exists(best): shutil.copy(best, '/kaggle/working/ckpt/chakra_transformer_best.pth')
for p in sorted(Path('/kaggle/working/ckpt').glob('*.pth')):
    print(f'{p.name:35s} {p.stat().st_size/1e6:8.0f} MB')